# `GaussianHermiteBasis`

An orthonormal basis of Hermite functions. They are square-integrable, so `Liouville` and
`KoopmanVonNeumann` accept it.

## 1. The basis functions

Products of Hermite **functions**, one factor per site:

$$\psi_n(v)=\frac{H_n(v)\,e^{-v^{2}/2}}{\sqrt{2^{n}\,n!\,\sqrt{\pi}}},
\qquad
\psi_{\boldsymbol\alpha}(u)=\prod_i \psi_{\alpha_i}(u_i),$$

with $H_n$ the physicists' Hermite polynomial (`sp.hermite`).

The $e^{-v^{2}/2}$ makes the set orthonormal and makes every $\psi_n$ decay. A bare monomial $u^{k}$ grows
without bound and carries no inner product.

## 2. The state class: `GaussianHermiteState`

`GaussianHermiteState` carries the index and nothing else, and prints that label. `(1, 2)` is
$\psi_2(u_1)$.

In [1]:
import sympy as sp

from lin_ode.basis import CarlemanBasis, GaussianHermiteBasis

gh = GaussianHermiteBasis(n_vars=3, degree=3)

print("gh[()]               ", gh[()])
print("gh[(1, 2)]           ", gh[(1, 2)])
print("gh[((0, 1), (2, 1))] ", gh[((0, 1), (2, 1))])

gh[()]                GaussianHermiteState()
gh[(1, 2)]            GaussianHermiteState((1, 2))
gh[((0, 1), (2, 1))]  GaussianHermiteState((0, 1), (2, 1))


## 3. The basis class: `GaussianHermiteBasis`

`GaussianHermiteBasis(n_vars, degree)` takes the same two arguments as `CarlemanBasis`, and gives the
same state count for a given site count and degree.

In [2]:
for basis in (CarlemanBasis(3, 2), GaussianHermiteBasis(3, 2)):
    print(f"{type(basis).__name__:22} len={len(basis):3}  n_dof={basis.n_dof}  "
          f"is_l2={basis.is_l2}")

CarlemanBasis          len= 10  n_dof=3  is_l2=False
GaussianHermiteBasis   len= 10  n_dof=3  is_l2=True


`is_l2 = True` says its functions are square-integrable. `Liouville` and `KoopmanVonNeumann` accept
this basis, and raise `IncompatibleBasisError` on either Carleman basis.

## 4. Indexing

The same structured and flat indices as every basis. `(1, 2)` means $\psi_2(u_1)$, and a site left
out of the label carries $\psi_0$.

In [3]:
basis = GaussianHermiteBasis(n_vars=2, degree=2)

for flat in range(len(basis)):
    structured = basis.structured_idx_of_flat_index(flat)
    print(f"  flat {flat}  <->  {str(structured):20}  {basis[structured]}")

print()
print("flat_index_of_structured_index((1, 2)) ", basis.flat_index_of_structured_index((1, 2)))
print("out of basis -> no flat index          ", basis.flat_index_of_basis_function(basis[(1, 3)]))

  flat 0  <->  ()                    GaussianHermiteState()
  flat 1  <->  ((0, 1),)             GaussianHermiteState((0, 1))
  flat 2  <->  ((1, 1),)             GaussianHermiteState((1, 1))
  flat 3  <->  ((0, 2),)             GaussianHermiteState((0, 2))
  flat 4  <->  ((0, 1), (1, 1))      GaussianHermiteState((0, 1), (1, 1))
  flat 5  <->  ((1, 2),)             GaussianHermiteState((1, 2))

flat_index_of_structured_index((1, 2))  5
out of basis -> no flat index           None


## 5. The computation rules

:::{note}
Sections 5 to 7 are how the library produces a value, not something you call.
`OperatorMatrix` and `Vector` apply these rules for you. Read them when you are adding a
basis of your own, or when an element is not the number you expected.
:::

The same two rules as every basis: multiplication by $u_s$, and $\partial/\partial u_s$. Here each
one is a ladder recurrence rather than a single term.

$$U_s:\;v\,\psi_n=\sqrt{\tfrac{n+1}{2}}\,\psi_{n+1}+\sqrt{\tfrac{n}{2}}\,\psi_{n-1},
\qquad
K_s:\;\psi_n'=-\sqrt{\tfrac{n+1}{2}}\,\psi_{n+1}+\sqrt{\tfrac{n}{2}}\,\psi_{n-1}.$$

Each returns two states instead of one, so a matrix element is a sum over ladder paths rather than a
lookup. Both move the index by exactly one, so a composition of $m$ of them reaches only indices
within $m$ of the start, and only those of the same parity as $m$.

In [4]:
from lin_ode.basis import GHKRule, GHURule

print("U on (site 0, n=2):", GHURule()(sp.Integer(0), sp.Integer(2)))
print("K on (site 0, n=2):", GHKRule()(sp.Integer(0), sp.Integer(2)))

U on (site 0, n=2): GaussianHermiteState((0, 1)) + sqrt(6)*GaussianHermiteState((0, 3))/2
K on (site 0, n=2): GaussianHermiteState((0, 1)) - sqrt(6)*GaussianHermiteState((0, 3))/2


## 6. Truncation

The cut is on the **Hermite index**. A state is a tuple of per-site orders $n_i$, and it is kept when
$\sum_i n_i\le$ degree. So "degree" counts Hermite orders, not polynomial powers.

In [5]:
basis = GaussianHermiteBasis(n_vars=2, degree=2)

for address in [(0, 1), (0, 2), (0, 3)]:
    print(f"  {str(address):10} ->", basis.truncation_rule(address))

  (0, 1)     -> GaussianHermiteState((0, 1))
  (0, 2)     -> GaussianHermiteState((0, 2))
  (0, 3)     -> 0


## 7. Expressions after truncation

`truncate_expression` does not truncate anything on this basis. It is `sp.expand`.

A polynomial in $u$ is not a combination of $\psi_n$, so there is nothing to drop until it is
projected.

In [6]:
basis = GaussianHermiteBasis(n_vars=2, degree=2)
u = sp.IndexedBase("u")

for expression in (u[0] ** 3, u[0] ** 2 * u[1], sp.exp(-u[0] ** 2 / 2)):
    print(f"  {str(expression):22} -> {basis.truncate_expression(expression, u)}")

print("\nall unchanged; it is sp.expand:",
      all(basis.truncate_expression(e, u) == sp.expand(e)
          for e in (u[0] ** 3, u[0] ** 2 * u[1], sp.exp(-u[0] ** 2 / 2))))

  u[0]**3                -> u[0]**3
  u[0]**2*u[1]           -> u[0]**2*u[1]
  exp(-u[0]**2/2)        -> exp(-u[0]**2/2)

all unchanged; it is sp.expand: True


Projection is an integral here, and truncation acts on the coefficients afterwards.
Whether that is safe depends on how fast they decay, which is a property of the function and not of
the basis.

$v^{2}$ is not in $L^{2}(\mathbb{R})$, so its Hermite coefficients do not decay and no finite sum
converges to it. A density's coefficients do decay.

So truncate a density, not a polynomial. A polynomial belongs on the other side of the inner
product, as an observable ([`06_vector`](../06_vector.ipynb)).

---

[← PSCarlemanBasis](psc_carleman.ipynb) · [Notebook index](../../README.md) · [4. The matrix →](../05_matrix.ipynb)